# Phase 1 Exploration

Minimal examples for poll aggregation, sentiment velocity, feature generation, and edge scoring.

In [ ]:
from datetime import UTC, datetime
from decimal import Decimal

from core import OrderBook, UnifiedMarket, Venue
from data.poll_aggregator import PollAggregator
from features import FeatureStore
from strategies import EdgeDetector

In [ ]:
polls = PollAggregator().normalize_records(
    [
        {
            "event_slug": "demo-event",
            "pollster": "Example Polls",
            "grade": "A",
            "sample_size": 1000,
            "end_date": "2026-05-01T00:00:00+00:00",
            "answers": [{"candidate": "Yes", "support": 55}, {"candidate": "No", "support": 45}],
        }
    ]
)
aggregates = PollAggregator().aggregate(polls, as_of=datetime(2026, 5, 12, tzinfo=UTC))
aggregates

In [ ]:
market = UnifiedMarket(
    venue=Venue.POLYMARKET, market_id="demo", title="Demo market", outcomes=["Yes", "No"]
)
order_book = OrderBook(
    market_id="demo",
    bids=[(Decimal("0.48"), Decimal("100"))],
    asks=[(Decimal("0.50"), Decimal("80"))],
)
detector = EdgeDetector(FeatureStore())
signal = await detector.score_market(
    market, model_context={"poll_aggregates": aggregates, "order_book": order_book}
)
signal